# ML - Aprendizaje No Supervisado

Si bien todos los ejemplos de *Machine Learning* que hemos visto hasta ahora se han basado en aprendizaje supervisado, la mayoría de los datos del mundo real **no están etiquetados**. Ante esta problemática podemos usar técnicas de **aprendizaje no supervisado** (*Unsupervised Learning*).

## Descripción del Dataset: Bank Customer Segmentation

### ¿De qué se trata?

El dataset **Bank Customer Segmentation** representa información de **10,000 clientes** de una entidad bancaria. Los datos capturan características demográficas, financieras y de comportamiento de cada cliente.

### ¿Qué contiene?

| Columna | Tipo | Descripción |
|---|---|---|
| **Edad** | Numérico | Edad del cliente (en años) |
| **Ingreso_Anual** | Numérico | Ingreso anual del cliente en $ |
| **Gasto_Mensual** | Numérico | Gasto mensual promedio en $ |
| **Ratio_Uso_Credito** | Numérico | Porcentaje de uso del límite de crédito (0-1) |
| **Num_Productos** | Numérico | Cantidad de productos bancarios contratados |
| **Limite_Credito** | Numérico | Límite de crédito total en $ |
| **Num_Transacciones** | Numérico | Número de transacciones mensuales |
| **Ratio_Deuda** | Numérico | Razón deuda/ingreso (0-1) |
| **Antiguedad_Anios** | Numérico | Años como cliente del banco |
| **Saldo_Promedio** | Numérico | Saldo promedio en cuenta en $ |

### ¿Para qué sirve?

El objetivo es aplicar **K-Means Clustering** para segmentar clientes en grupos con comportamiento financiero similar. Dado que trabajamos con **10 dimensiones**, usaremos **PCA** para reducir a 2D y poder visualizar los clusters.

Al aplicar K-Means con k=5, el algoritmo descubre estos 5 segmentos:

| Grupo | Perfil del cliente |
|---|---|
| Cluster 0 | Jóvenes con bajo uso financiero |
| Cluster 1 | Clientes VIP - alto consumo |
| Cluster 2 | Clientes promedio estables |
| Cluster 3 | Alto ingreso, bajo gasto |
| Cluster 4 | Bajo ingreso, alto endeudamiento |


## 1. Carga y exploración del Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Carga del dataset
df = pd.read_csv('bank_customers.csv')

print("Primeras filas del dataset:")
print(df.head())
print(f"\nDimensiones: {df.shape[0]} clientes, {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
print("\nEstadísticas descriptivas:")
print(df.describe())


In [ ]:
# Separamos las features (descartamos la columna Cluster_Real que solo existe para validación)
X = df.drop('Cluster_Real', axis=1).values
feature_names = df.drop('Cluster_Real', axis=1).columns.tolist()

print("Shape de X:", X.shape)
print("Features:", feature_names)


## 2. Preprocesamiento: Escalado de características

Con 10 features de distintas escalas (edad en años, ingreso en miles de $, ratios entre 0 y 1), es **imprescindible** normalizar antes de aplicar K-Means. Sin escalado, features con valores grandes dominarían el cálculo de distancias.

Usamos `StandardScaler` para que cada feature tenga **media 0 y desviación estándar 1**.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Antes del escalado:")
print(f"  Ingreso_Anual - media: {X[:, 1].mean():.0f}, std: {X[:, 1].std():.0f}")
print(f"  Ratio_Deuda   - media: {X[:, 7].mean():.3f}, std: {X[:, 7].std():.3f}")

print("\nDespués del escalado:")
print(f"  Ingreso_Anual - media: {X_scaled[:, 1].mean():.3f}, std: {X_scaled[:, 1].std():.3f}")
print(f"  Ratio_Deuda   - media: {X_scaled[:, 7].mean():.3f}, std: {X_scaled[:, 7].std():.3f}")


## 3. Reducción de Dimensionalidad con PCA

Como tenemos 10 dimensiones, no podemos visualizar directamente. Aplicamos **PCA (Análisis de Componentes Principales)** para proyectar los datos en 2D manteniendo la mayor varianza posible.

> **Nota:** PCA se aplica **solo para visualización**. El clustering se realiza sobre los 10 features originales escalados.


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza explicada por componente:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.1f}%")

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.4, color='steelblue')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)", fontsize=13)
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)", fontsize=13)
plt.title("Clientes Bancarios en espacio PCA (sin etiquetas)", fontsize=14)
plt.tight_layout()
plt.show()


## 4. Método del Codo para elegir k óptimo

Antes de aplicar K-Means necesitamos determinar el número de clusters **k**. El **método del codo** (*Elbow Method*) consiste en graficar la inercia (suma de distancias al centroide) para distintos valores de k y buscar el punto donde la mejora se vuelve marginal.


In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=2000, random_state=42)
    silhouette_scores.append(sil)
    print(f"k={k}: inercia={km.inertia_:.0f}, silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=5, color='red', linestyle='--', label='k=5 (óptimo)')
axes[0].set_xlabel("Número de clusters k", fontsize=13)
axes[0].set_ylabel("Inercia", fontsize=13)
axes[0].set_title("Método del Codo", fontsize=14)
axes[0].legend()

# Silhouette Score
axes[1].plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].axvline(x=5, color='blue', linestyle='--', label='k=5 (óptimo)')
axes[1].set_xlabel("Número de clusters k", fontsize=13)
axes[1].set_ylabel("Silhouette Score", fontsize=13)
axes[1].set_title("Silhouette Score por k", fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. K-Means Clustering

Aplicamos K-Means con **k=5** sobre los datos escalados en 10 dimensiones.


In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_scaled)

print(f"Silhouette Score final: {silhouette_score(X_scaled, y_pred):.4f}")
print(f"\nDistribución de clientes por cluster:")
unique, counts = np.unique(y_pred, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  Cluster {c}: {n} clientes ({n/len(y_pred)*100:.1f}%)")


## 6. Visualización de Clusters en espacio PCA

In [ ]:
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
labels = ['Jóvenes bajo uso', 'Clientes VIP', 'Promedio estables',
          'Alto ingreso bajo gasto', 'Bajo ingreso alto endeud.']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Sin etiquetas ---
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.3, color='gray')
axes[0].set_title("Clientes sin etiquetas (PCA)", fontsize=13)
axes[0].set_xlabel("PC1", fontsize=12)
axes[0].set_ylabel("PC2", fontsize=12)

# --- Con clusters K-Means ---
for i in range(k):
    mask = y_pred == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=8, alpha=0.5, c=colors[i], label=f'Cluster {i}: {labels[i]}')

# Centroides proyectados en PCA
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[1].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                s=250, c='yellow', edgecolors='black', linewidths=2,
                marker='*', zorder=10, label='Centroides')

axes[1].set_title("K-Means Clustering (k=5) proyectado en PCA", fontsize=13)
axes[1].set_xlabel("PC1", fontsize=12)
axes[1].set_ylabel("PC2", fontsize=12)
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()


## 7. Perfil de cada Cluster

Analizamos los centroides para interpretar qué tipo de cliente representa cada grupo.


In [ ]:
# Centroides en escala original
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroids = pd.DataFrame(centroids_original, columns=feature_names)
df_centroids.index = [f'Cluster {i}' for i in range(k)]

print("Perfil promedio de cada cluster (escala original):")
print(df_centroids.round(1).to_string())


In [ ]:
# Heatmap de centroides normalizado
fig, ax = plt.subplots(figsize=(12, 4))

centroids_norm = (df_centroids - df_centroids.min()) / (df_centroids.max() - df_centroids.min())
im = ax.imshow(centroids_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(k))
ax.set_yticklabels([f'Cluster {i}: {labels[i]}' for i in range(k)], fontsize=10)

plt.colorbar(im, ax=ax, label='Valor normalizado (0=min, 1=max)')
plt.title("Heatmap de perfiles de Clusters", fontsize=14)
plt.tight_layout()
plt.show()


## 8. Predicción de nuevos clientes

Una vez entrenado el modelo, podemos clasificar nuevos clientes directamente.


In [ ]:
# Nuevos clientes a clasificar
nuevos_clientes = np.array([
    [22, 18000, 200,  0.05, 1, 2000,  3, 0.1, 0,  150],   # Joven reciente
    [48, 92000, 7500, 0.75, 11, 14000, 48, 0.7, 9, 2900],  # Posible VIP
    [33, 44000, 2800, 0.42, 6,  5800,  19, 0.5, 4, 1100],  # Promedio
])

nuevos_scaled = scaler.transform(nuevos_clientes)
clusters_pred = kmeans.predict(nuevos_scaled)

print("Clasificación de nuevos clientes:")
for i, (cliente, cluster) in enumerate(zip(nuevos_clientes, clusters_pred)):
    print(f"  Cliente {i+1} -> Cluster {cluster}: {labels[cluster]}")


## 9. Fronteras de Decisión en espacio PCA

Visualizamos cómo K-Means divide el espacio PCA en regiones de decisión.


In [ ]:
resolution = 500
mins = X_pca.min(axis=0) - 0.5
maxs = X_pca.max(axis=0) + 0.5
xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                     np.linspace(mins[1], maxs[1], resolution))

# Invertimos PCA para predecir en espacio original
grid_pca = np.c_[xx.ravel(), yy.ravel()]
grid_original = pca.inverse_transform(grid_pca)
Z = kmeans.predict(grid_original)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, cmap='Pastel1', alpha=0.7)
plt.contour(xx, yy, Z, colors='k', linewidths=0.5)

for i in range(k):
    mask = y_pred == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                s=6, alpha=0.4, c=colors[i], label=f'Cluster {i}: {labels[i]}')

plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            s=300, c='yellow', edgecolors='black', linewidths=2,
            marker='*', zorder=10, label='Centroides')

plt.xlabel("PC1", fontsize=13)
plt.ylabel("PC2", fontsize=13)
plt.title("K-Means — Fronteras de decisión (espacio PCA)", fontsize=14)
plt.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()
